In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import ConnectionPatch

In [ ]:
results_dir = '../exp_results'

vanilla_results = pd.read_csv(f'{results_dir}/vanilla_t5tiny/results.csv')
mica_results = pd.read_csv(f'{results_dir}/infini_mlpquerymixer_t5tiny/results.csv')
itransformer_results = pd.read_csv(f'{results_dir}/itransformer_baseline/results.csv')
timerxl_results = pd.read_csv(f'{results_dir}/timerxl_baseline/results.csv')
crossformer_results = pd.read_csv(f'{results_dir}/crossformer_baseline/results.csv')
tsmixer_results = pd.read_csv(f'{results_dir}/tsmixer_baseline/results.csv')
timemixer_results = pd.read_csv(f'{results_dir}/timemixer_baseline/results.csv')
mlp_results = pd.read_csv(f'{results_dir}/multivariateMLP_baseline/results.csv')
chronos_results = pd.read_csv(f'{results_dir}/chronos2.0_baseline/results.csv')

flops = pd.read_csv('../tables/flops_baseline_table.csv')
flops.set_index('model', inplace=True)

flops2 = pd.read_csv('../tables/flops_mica_table.csv')
patchtst_flops2 = flops2.iloc[:8]
patchtst_flops2.set_index('variant', inplace=True)
moment_flops2 = flops2.iloc[8:]
moment_flops2.set_index('variant', inplace=True)

In [ ]:
table = pd.concat([
    vanilla_results[['dataset', 'AutoMOMENT_vanilla_mae_mean']].set_index('dataset'),
    vanilla_results[['dataset', 'AutoPatchTSTMultivariate_vanilla_mae_mean']].set_index('dataset'),
    mica_results[['dataset', 'AutoMOMENT_mlpquerymixer_ciincl_mae_mean']].set_index('dataset'),
    mica_results[['dataset', 'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_mae_mean']].set_index('dataset'),
    itransformer_results[['dataset', 'AutoiTransformer_multivariate_mae_mean']].set_index('dataset'),
    itransformer_results[['dataset', 'AutoiTransformerT5_multivariate_mae_mean']].set_index('dataset'),
    timerxl_results[['dataset', 'AutoTimerXL_multivariate_mae_mean']].set_index('dataset'),
    crossformer_results[['dataset', 'AutoCrossformer_multivariate_mae_mean']].set_index('dataset'),
    tsmixer_results[['dataset', 'AutoTSMixer_multivariate_mae_mean']].set_index('dataset'),   
    timemixer_results[['dataset', 'AutoTimeMixer_multivariate_mae_mean']].set_index('dataset'),   
    mlp_results[['dataset', 'AutoMLPMultivariate_multivariate_mae_mean']].set_index('dataset'),
    chronos_results[['dataset', 'Chronos_multivariate_mae_mean']].set_index('dataset')
    ], axis=1
)
table.rename(columns={
    'AutoMOMENT_vanilla_mae_mean':'Moment',
    'AutoPatchTSTMultivariate_vanilla_mae_mean':'PatchTST',
    'AutoMOMENT_mlpquerymixer_ciincl_mae_mean':'Moment-MICA',
    'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_mae_mean':'PatchTST-MICA',
    'AutoiTransformer_multivariate_mae_mean':'iTransformer',
    'AutoiTransformerT5_multivariate_mae_mean':'iTransformerT5',
    'AutoTimerXL_multivariate_mae_mean':'Timer-XL',
    'AutoCrossformer_multivariate_mae_mean':'Crossformer',
    'AutoTSMixer_multivariate_mae_mean':'TSMixer',
    'AutoTimeMixer_multivariate_mae_mean':'TimeMixer',
    'AutoMLPMultivariate_multivariate_mae_mean':'MLP',
    'Chronos_multivariate_mae_mean':'Chronos-2',
}, inplace=True)

table = table.rank(axis=1, method='min')
table = table.mean(axis=0)

In [ ]:
models = {
    'Moment': {
        'error': table['Moment'],
        'flops': moment_flops2.loc['Univariate']['gflops'], 
        'params': moment_flops2.loc['Univariate']['trainable_params']
    },
    'PatchTST': {
        'error': table['PatchTST'],
        'flops': patchtst_flops2.loc['Univariate']['gflops'],
        'params': patchtst_flops2.loc['Univariate']['trainable_params']
    },
    'Moment-MICA': {
        'error': table['Moment-MICA'],
        'flops': moment_flops2.loc['MICA (MLP w/ Query)']['gflops'], 
        'params': moment_flops2.loc['MICA (MLP w/ Query)']['trainable_params']
    },
    'PatchTST-MICA': {
        'error': table['PatchTST-MICA'],
        'flops': patchtst_flops2.loc['MICA (MLP w/ Query)']['gflops'],
        'params': patchtst_flops2.loc['MICA (MLP w/ Query)']['trainable_params']
    },
    'iTranformer': {
        'error': table['iTransformer'],
        'flops': flops.loc['iTransformer']['gflops'], 
        'params': flops.loc['iTransformer']['trainable_params']
    },
    'iTranformerT5': {
        'error': table['iTransformerT5'],
        'flops': flops.loc['iTransformer']['gflops'], 
        'params': flops.loc['iTransformer']['trainable_params']
    },
    'Crossformer': {
        'error': table['Crossformer'],
        'flops': flops.loc['Crossformer']['gflops'], 
        'params': flops.loc['Crossformer']['trainable_params']
    },
    'Timer-XL': {
        'error': table['Timer-XL'],
        'flops': flops.loc['Timer-XL']['gflops'], 
        'params': flops.loc['Timer-XL']['trainable_params']
    },
    'TSMixer': {
        'error': table['TSMixer'],
        'flops': flops.loc['TSMixer']['gflops'], 
        'params': flops.loc['TSMixer']['trainable_params']
    },
    'TimeMixer': {
        'error': table['TimeMixer'],
        'flops': flops.loc['TimeMixer']['gflops'], 
        'params': flops.loc['TimeMixer']['trainable_params']
    },
    'MLP': {
        'error': table['MLP'],
        'flops': flops.loc['MLP']['gflops'], 
        'params': flops.loc['MLP']['trainable_params']
    },
    'Chronos-2': {
        'error': table['Chronos-2'],
        'flops': flops.loc['Chronos-2']['gflops'], 
        'params': flops.loc['Chronos-2']['trainable_params']
    },
}

In [ ]:
flops = [data['flops'] for data in models.values()]
performance = [data['error'] for data in models.values()]
params = [data['params'] for data in models.values()]
names = list(models.keys())

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
fontsize = 22

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5),
                                gridspec_kw={'width_ratios': [3, 0.5]})

# Normalize parameter counts for circle sizes
size_scale = 3000
sizes = [np.sqrt(p) * size_scale / np.sqrt(max(params)) for p in params]

# colormap
original_cmap = plt.cm.Blues
colors = original_cmap(np.linspace(0.3, 1.0, 256))
truncated_cmap = LinearSegmentedColormap.from_list('Blues_truncated', colors)

for ax in [ax1, ax2]:
    scatter = ax.scatter(flops, performance, s=sizes, alpha=0.6,
                         c=params, cmap=truncated_cmap,
                         edgecolors='black', linewidth=1)

ax1.set_xlim(-0.1, 2.7)
ax2.set_xlim(15.9, 17.1)
ax1.set_ylim([0, 10])
ax2.set_ylim([0, 10])

ax1.spines['right'].set_visible(False)
ax2.spines['left'].set_visible(False)

ax2.yaxis.set_tick_params(left=False, labelleft=False)
ax2.set_axisbelow(True)
ax2.grid(True, axis='y')

# Diagonal break marks
d = 0.015
kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False, linewidth=2.5)
ax1.plot((1-d, 1+d), (-d, +d), **kwargs)
ax1.plot((1-d, 1+d), (1-d, 1+d), **kwargs)

kwargs.update(transform=ax2.transAxes)
ax2.plot((-d, +d), (-d, +d), **kwargs)
ax2.plot((-d, +d), (1-d, 1+d), **kwargs)

ax1.tick_params(axis="x", labelsize=fontsize-2, colors='black')
ax1.tick_params(axis="y", labelsize=fontsize-2, colors='black')
ax2.tick_params(axis="x", labelsize=fontsize-2, colors='black')

ax1.set_ylabel('Average Rank', fontsize=fontsize)
ax1.set_xlabel('')
ax2.set_xlabel('')


label_positions = {
    'PatchTST':       (18,  -5,   'left',   'center'),
    'PatchTST-MICA':  (-10, -22,  'left',   'center'),
    'Moment':         (55, 0,  'center', 'bottom'),
    'Moment-MICA':    (84,  0,  'center', 'bottom'),
    'MLP':            (5, 24,  'center', 'top'),
    'TSMixer':        (18,    7,  'center', 'bottom'),
    'TimeMixer':      (6,   14,  'center', 'bottom'),
    'iTranformerT5':  (44,  -15,  'center', 'top'),
    'iTranformer':    (65,   20,  'center', 'top'),
    'Timer-XL':       (44, 15,  'center', 'bottom'),
    'Crossformer':    (-3,  20,  'center', 'bottom'),
}
label_positions_ax2 = {
    'Chronos-2': (-44, 38, 'left', 'center'),  # increased y_off from 0 to 20
}

for i, name in enumerate(names):
    if name == 'Chronos-2':
        x_off, y_off, ha, va = label_positions_ax2.get(name, (0, 10, 'left', 'center'))
        ax2.annotate(name, (flops[i], performance[i]),
                     xytext=(x_off, y_off), textcoords='offset points',
                     fontsize=fontsize-6, ha=ha, va=va,
                     fontweight='bold')
    else:
        if name in label_positions:
            x_off, y_off, ha, va = label_positions[name]
        else:
            x_off, y_off, ha, va = 0, 10, 'center', 'bottom'
        ax1.annotate(name, (flops[i], performance[i]),
                     xytext=(x_off, y_off), textcoords='offset points',
                     fontsize=fontsize-6, ha=ha, va=va,
                     fontweight='bold')

norm = plt.Normalize(vmin=min(params), vmax=max(params))
legend_params = [min(params), np.median(params), max(params)]
legend_labels = [f'{p:.0f}M' if p < 1e9 else f'{p/1e9:.1f}B' for p in legend_params]
legend_sizes = [np.sqrt(p) * size_scale / np.sqrt(max(params)) for p in legend_params]
legend_colors = [truncated_cmap(norm(p)) for p in legend_params]
legend_elements = [plt.scatter([], [], s=s, c=[color], alpha=0.6, edgecolors='black',
                               linewidth=1, label=label)
                   for s, label, color in zip(legend_sizes, legend_labels, legend_colors)]
legend = ax2.legend(handles=legend_elements, title='Trainable \nParameters',
                    bbox_to_anchor=(1., 1.04), frameon=True, fontsize=fontsize-2,
                    ncol=1, labelspacing=2.5, title_fontsize=fontsize-2,
                    borderpad=1.1,        # reduced from 1.2
                    handletextpad=1.0,    # reduced from 1.5
                    scatteryoffsets=[0.5])

plt.tight_layout()
plt.subplots_adjust(wspace=0.05, bottom=0.18)
fig.text(0.5, 0.04, 'GFLOPs', ha='center', fontsize=fontsize)

plt.savefig('./flops_comparison_main.pdf')
